# TarMAC

> TarMAC (Das et al., 2019): recurrent agents that send messages with a signature and a value, which receivers attend to with a query; multi-round communication; trained online with the paper's multi-agent actor-critic (centralized critic).

In [ ]:
#| default_exp wm.tarmac.core

In [ ]:
#| hide
from nbdev.showdoc import *

[TarMAC](https://arxiv.org/abs/1810.11187) ("TarMAC: Targeted Multi-Agent Communication", Das,
Gervet, Romoff, Batra, Parikh, Rabbat and Pineau, ICML 2019) makes communication **targeted**:

* every agent is a GRU (shared by the agents) over its observation `ω_i^t` and the aggregated
  message `c_i^t` it received: `h_i^t = GRU([enc(ω_i^t), c_i^t], h_i^{t-1})`; its action comes from
  `h_i^t`, and so does its message `m_i^t = [k_i^t, v_i^t]`, a **signature** `k` (`key_dim`, 16) and a
  **value** `v` (`value_dim`, 32), sent to the others;
* a receiver `j` predicts a **query** `q_j` and attends to all the messages, its own included
  (self-attention): `α_ji = softmax_i(q_j · k_i / sqrt(key_dim))`, `c_j^{t+1} = Σ_i α_ji v_i^t`, its input
  at the next step (messages arrive one step later);
* with `rounds > 1`, `h'_j = tanh(W [c_j ‖ h_j])` predicts new signatures, queries and values, and the
  attention repeats; the last round's `c` is the next step's input.

The paper's Eq. 2 writes the query as `q_j^{t+1}` from `h_j^{t+1}`, while Eq. 4 combines
`c_j^{t+1}` with `h_j^t`; here the query is predicted from `h_j^t` (and `h'_j` in later rounds), as
in Eq. 4, so that `c^{t+1}` is ready before step `t+1`.

**Training** is the paper's multi-agent actor-critic, batched and synchronous: a **centralized
critic** `Q̂(h_1^t, ..., h_N^t, a_1^t, ..., a_N^t)` of the team's return (used only in training,
learned by temporal differences), and the policy gradient
`∇ log π(a_i^t | h_i^t) Q̂_t` (no baseline subtracted, as in the paper; `q_baseline` subtracts the
batch's mean `Q̂`, not in the paper), with an entropy bonus (0.01), RMSProp (lr `7e-4`, alpha
0.99), `gamma = 0.99`, and batches of 16 episodes (one per env of the `World`). Each update replays
its episodes through the recurrent policy from their start (backpropagation through time) with the
communication masks recorded when acting. The reward is the team's (the sum of the agents'
rewards).

As in the other baselines, agents that reached their goal act no more and leave the communication
(they are neither heard nor counted by the critic), the observations are `commnet_features`, and
messages can go over comm-core networks (`links`, `NetworkLinks`: one bundle `[k, v]` per agent
and round; `latent_bits` quantizes them with comm-core's `UniformQuantizer`, after a tanh).

In [ ]:
#| export
from __future__ import annotations

import math
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from fastcore.utils import patch

from stable_marl.wm.commnet.core import NetworkLinks, comm_mask, commnet_features
from stable_marl.world.env_pool import episode_done

### Targeted communication

In [ ]:
#| export
def targeted_attention(
    query: torch.Tensor, # (B, N, key_dim) of the receivers
    key: torch.Tensor, # (B, N, key_dim) signatures of the senders
    value: torch.Tensor, # (B, N, value_dim)
    hears: torch.Tensor | None = None, # (B, N, N) bool: receiver i got sender k's message (its own: always); None: all
) -> tuple[torch.Tensor, torch.Tensor]:
    "Aggregated messages (B, N, value_dim) and attention weights (B, N, N): softmax over the messages heard, self included."
    B, N, d = key.shape
    scores = query @ key.transpose(1, 2) / math.sqrt(d)
    if hears is not None:
        heard = hears.bool() | torch.eye(N, dtype=torch.bool, device=key.device)
        scores = scores.masked_fill(~heard, float('-inf'))
    weights = torch.softmax(scores, -1)
    return weights @ value, weights

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail
torch.manual_seed(0)
q, k, v = torch.randn(2, 3, 4), torch.randn(2, 3, 4), torch.randn(2, 3, 5)
everyone = torch.ones(2, 3, 3, dtype=torch.bool)
c, w = targeted_attention(q, k, v, everyone)
test_close(c, targeted_attention(q, k, v)[0])                          # no mask: everyone
test_close(w.sum(-1), torch.ones(2, 3)); test_close(c, torch.softmax(q @ k.transpose(1, 2) / 2, -1) @ v)
nobody = torch.zeros(2, 3, 3, dtype=torch.bool)
c, w = targeted_attention(q, k, v, nobody)
test_close(c, v); test_close(w, torch.eye(3).expand(2, 3, 3))         # only its own message: self-attention
lost = everyone.clone(); lost[:, 0, 1] = False
_, w = targeted_attention(q, k, v, lost)
test_eq(w[:, 0, 1], torch.zeros(2))                                    # a lost message gets no weight

### The agents and the critic

In [ ]:
#| export
class TarMAC(nn.Module):
    "TarMAC's recurrent agents (shared by the agents) and its centralized critic."
    def __init__(
        self,
        obs_dim: int, # Features per agent (see `commnet_features`)
        num_agents: int,
        num_actions: int = 4,
        hidden_dim: int = 128, # The GRU's hidden state (TarMAC: 128)
        key_dim: int = 16, # Signatures and queries (TarMAC: 16)
        value_dim: int = 32, # Message values (TarMAC: 32)
        rounds: int = 1, # Rounds of communication per step (TarMAC: 1 or 2)
        critic_hidden: int = 128, # Width of the critic's MLP
        obs_keys: Sequence[str] = ('image', 'direction'),
        goal: bool = True, # Observe the offset to the goal cell
        grid_size: int = 15,
        comm_range: int | None = None, # Agents hear the agents within this many cells (None: all)
        drop_prob: float = 0.0, # Probability that a message is lost (each pair and round)
        latent_bits: int | None = None, # Bits per number of the messages (tanh, then comm-core's UniformQuantizer); None: float32
    ):
        super().__init__()
        if rounds < 1:
            raise ValueError(f"rounds must be >= 1, got {rounds}")
        self.config = dict(obs_dim=obs_dim, num_agents=num_agents, num_actions=num_actions, hidden_dim=hidden_dim,
                           key_dim=key_dim, value_dim=value_dim, rounds=rounds, critic_hidden=critic_hidden,
                           obs_keys=list(obs_keys), goal=goal, grid_size=grid_size, comm_range=comm_range,
                           drop_prob=drop_prob, latent_bits=latent_bits)
        self.num_agents, self.num_actions, self.hidden_dim, self.value_dim = num_agents, num_actions, hidden_dim, value_dim
        self.rounds, self.obs_keys, self.goal, self.grid_size = rounds, tuple(obs_keys), goal, grid_size
        self.comm_range, self.drop_prob, self.latent_bits = comm_range, drop_prob, latent_bits
        self.encoder = nn.Sequential(nn.Linear(obs_dim, hidden_dim), nn.ReLU())
        self.gru = nn.GRUCell(hidden_dim + value_dim, hidden_dim)
        self.policy_head = nn.Linear(hidden_dim, num_actions)
        self.signature = nn.Linear(hidden_dim, key_dim)
        self.query = nn.Linear(hidden_dim, key_dim)
        self.value = nn.Linear(hidden_dim, value_dim)
        self.merge = nn.Linear(value_dim + hidden_dim, hidden_dim)     # W_{h -> h'} of the later rounds
        self.critic = nn.Sequential(nn.Linear(num_agents * (hidden_dim + num_actions + 1), critic_hidden), nn.ReLU(),
                                    nn.Linear(critic_hidden, critic_hidden), nn.ReLU(), nn.Linear(critic_hidden, 1))
        self.temperature = 0.0     # acting: 0 picks the most likely action, otherwise samples
        self.checkpoint: dict = {}
        self.links: NetworkLinks | None = None
        self._quantizer = None
        self._state: tuple[torch.Tensor, torch.Tensor] | None = None   # acting: (h, c) of every env and agent

    @property
    def message_bits(self) -> int:
        "Bits of a message `[k, v]` (what each agent sends per round)."
        return (self.config['key_dim'] + self.value_dim) * (self.latent_bits or 32)

    def _send(self, x: torch.Tensor) -> torch.Tensor:
        "A message part as received: tanh and `UniformQuantizer` with `latent_bits` (straight-through), or as it is."
        if self.latent_bits is None:
            return x
        if self._quantizer is None:
            from comm_core import UniformQuantizer
            self._quantizer = UniformQuantizer(self.latent_bits, -1.0, 1.0)
        x = torch.tanh(x)
        return x + (self._quantizer.decode(self._quantizer.encode(x.detach())).to(x.dtype) - x).detach()

    def step(
        self,
        obs: torch.Tensor, # (B, N, obs_dim)
        alive: torch.Tensor, # (B, N) bool: agents acting (the others are reset and silent)
        hears: torch.Tensor, # (B, rounds, N, N) bool: who got whose message, each round
        h: torch.Tensor, # (B, N, hidden_dim) the previous hidden states
        c: torch.Tensor, # (B, N, value_dim) the messages aggregated at the previous step
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        "One step of every agent: action logits (B, N, num_actions), the new hidden states, and the messages for the next step."
        live = alive.unsqueeze(-1).to(obs.dtype)
        B, N, _ = obs.shape
        h = self.gru(torch.cat([self.encoder(obs), c], -1).flatten(0, 1), h.flatten(0, 1)).view(B, N, -1) * live
        logits = self.policy_head(h)
        state = h
        for r in range(self.rounds):
            heard = hears[:, r] & alive.unsqueeze(1)                    # finished agents send nothing
            c, _ = targeted_attention(self.query(state), self._send(self.signature(state)), self._send(self.value(state)), heard)
            if r < self.rounds - 1:
                state = torch.tanh(self.merge(torch.cat([c, state], -1)))
        return logits, h, c * live

    def q_value(self, h: torch.Tensor, action: torch.Tensor, alive: torch.Tensor) -> torch.Tensor:
        "The centralized critic Q̂ (B,) of every agent's hidden state and action (agents not acting: zeros)."
        live = alive.unsqueeze(-1).float()
        x = torch.cat([h.detach() * live, F.one_hot(action.long(), self.num_actions).float() * live, live], -1)
        return self.critic(x.flatten(1)).squeeze(-1)

    def initial_state(self, B: int, device=None) -> tuple[torch.Tensor, torch.Tensor]:
        "Hidden states and aggregated messages at the start of an episode (zeros)."
        return (torch.zeros(B, self.num_agents, self.hidden_dim, device=device),
                torch.zeros(B, self.num_agents, self.value_dim, device=device))

In [ ]:
#| hide
torch.manual_seed(0)
torch.set_grad_enabled(False)
B, N, D = 2, 3, 6
net = TarMAC(D, N, hidden_dim=16)
alive = torch.ones(B, N, dtype=torch.bool)
hears = torch.ones(B, 1, N, N, dtype=torch.bool)
obs = torch.randn(B, N, D)
other = obs.clone(); other[:, 1] += 1
h, c = net.initial_state(B)
logits, h1, c1 = net.step(obs, alive, hears, h, c)
logits_other, h1o, c1o = net.step(other, alive, hears, h, c)
test_close(logits[:, 0], logits_other[:, 0])                          # agent 1's step-t observation does not reach agent 0 at t
assert not torch.allclose(c1[:, 0], c1o[:, 0])                        # but its message does, for t + 1
assert not torch.allclose(net.step(obs, alive, hears, h1, c1)[0][:, 0], net.step(obs, alive, hears, h1o, c1o)[0][:, 0])
deaf = torch.zeros(B, 1, N, N, dtype=torch.bool)
test_close(net.step(obs, alive, deaf, h, c)[2][:, 0], net.step(other, alive, deaf, h, c)[2][:, 0])   # no messages heard
two = TarMAC(D, N, hidden_dim=16, rounds=2)
test_eq(two.step(obs, alive, torch.ones(B, 2, N, N, dtype=torch.bool), h, c)[2].shape, (B, N, 32))
dead = alive.clone(); dead[:, 2] = False
_, h2, c2 = net.step(obs, dead, hears, h, c)
test_eq(h2[:, 2].abs().sum(), torch.tensor(0.)); test_eq(c2[:, 2].abs().sum(), torch.tensor(0.))
test_eq(net.q_value(h1, torch.zeros(B, N), alive).shape, (B,))
test_eq(TarMAC(D, N, latent_bits=4).message_bits, 48 * 4)
test_fail(lambda: TarMAC(D, N, rounds=0))
torch.set_grad_enabled(True)

### Observing and acting

`observe` turns columns of one step (`(B, N, ...)`) into the agents' features, the agents still
acting, and who hears whom in each round (the alive agents, `comm_range`, `drop_prob`, and with
`links` the comm-core networks, which send each round's `[k, v]` of `message_bits`). As the model of
a `FeedForwardPolicy`, `get_action` keeps every env's hidden states and messages, restarted when it
resets (`_needs_flush`).

In [ ]:
#| export
@patch
def _info_keys(self: TarMAC) -> list[str]:
    keys = [*self.obs_keys, 'terminated', 'position']
    if self.goal:
        keys.append('goal_position')
    return list(dict.fromkeys(keys))


@patch
def observe(self: TarMAC, columns: dict[str, np.ndarray]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    "Features (B, N, obs_dim), `alive` (B, N) and `hears` (B, rounds, N, N) of columns of one step."
    obs = torch.from_numpy(commnet_features(columns, self.obs_keys, self.grid_size, self.goal))
    alive = ~np.asarray(columns['terminated'], bool)
    position = np.asarray(columns['position'])
    hears = torch.stack([comm_mask(torch.from_numpy(alive), torch.as_tensor(position) if self.comm_range is not None else None,
                                   self.comm_range, self.drop_prob) for _ in range(self.rounds)], 1)
    if self.links is not None:
        hears = hears & self.links.masks(position, alive, self.rounds, 0, message_bits=[self.message_bits] * self.rounds)
    return obs * torch.from_numpy(alive).unsqueeze(-1), torch.from_numpy(alive), hears


@patch
@torch.no_grad()
def get_action(self: TarMAC, info: dict) -> torch.Tensor:
    "Actions (num_envs, num_agents) for the info dict of a pool of envs, carrying every env's hidden states and messages."
    columns = {k: torch.as_tensor(info[k])[:, -1].cpu().numpy() for k in self._info_keys()}
    E = len(columns['terminated'])
    device = next(self.parameters()).device
    flush = info.get('_needs_flush')
    flush = np.ones(E, bool) if flush is None else torch.as_tensor(flush).cpu().numpy().astype(bool)
    if self._state is None or len(self._state[0]) != E or self._state[0].device != device:
        self._state, flush = self.initial_state(E, device), np.ones(E, bool)
    if self.links is not None and flush.any():
        seeds = torch.as_tensor(info['seed'])[:, -1].cpu().numpy() if 'seed' in info else None
        envs = np.flatnonzero(flush)
        self.links.reset(envs, None if seeds is None else [int(seeds[e]) for e in envs])
    h, c = self._state
    restart = torch.from_numpy(flush).to(device)
    h, c = h.masked_fill(restart[:, None, None], 0.0), c.masked_fill(restart[:, None, None], 0.0)
    obs, alive, hears = self.observe(columns)
    logits, h, c = self.step(obs.to(device), alive.to(device), hears.to(device), h, c)
    self._state = (h, c)
    if self.temperature > 0:
        return torch.distributions.Categorical(logits=logits / self.temperature).sample()
    return logits.argmax(-1)


@patch
def reset(self: TarMAC):
    "Forget the hidden states and messages of the last episodes."
    self._state = None

### Rollouts

`rollout` plays one episode in each env of a `World`, recording every step's inputs (features,
`alive`, `hears`), the sampled actions and the team reward. When an env's episode ends at the time
limit with agents still on their way, one more row holds its last state and actions sampled there
(`bootstrap`): the critic's value of it is the TD target's continuation. The rows after an env's
end are padding (nobody acting).

In [ ]:
#| export
def rollout(model: TarMAC, world, seed: int) -> tuple[dict[str, torch.Tensor], dict[str, np.ndarray]]:
    """
    One episode per env, as rows (L, E, ...): `obs`, `alive`, `hears`, `action`, `log_prob`, `acting` (the env was
    stepped with this row's actions), `reward` (L, E: the team's), `continues` (L, E: the next row
    continues the episode, or bootstraps it); per env `success`, `return` and `length`.
    """
    E, N = world.num_envs, world.num_agents
    world.reset(seed=[seed + i for i in range(E)])
    if model.links is not None:
        model.links.reset(range(E), [seed + i for i in range(E)])
    device = next(model.parameters()).device
    h, c = model.initial_state(E, device)
    running, bootstrap = np.ones(E, bool), np.zeros(E, bool)          # bootstrap: the next row is the env's last state
    reached, length = np.zeros((E, N), bool), np.zeros(E, np.int64)
    rows: dict[str, list] = {k: [] for k in ('obs', 'alive', 'hears', 'action', 'log_prob', 'acting', 'reward', 'continues')}
    while running.any() or bootstrap.any():
        columns = {k: np.asarray(world.infos[k])[:, -1].copy() for k in model._info_keys()}
        columns['terminated'] = columns['terminated'] | ~(running | bootstrap)[:, None]
        obs, alive, hears = model.observe(columns)
        with torch.no_grad():
            logits, h, c = model.step(obs.to(device), alive.to(device), hears.to(device), h, c)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            log_prob, action = dist.log_prob(action).cpu(), action.cpu()
        stepped = running.copy()
        _, rewards, terms, truncs, world.infos = world.envs.step(action.numpy(), mask=stepped)
        ended = stepped & episode_done(terms, truncs)
        cut = ended & ~terms.all(1)                                     # the time limit, agents still on their way
        reward = (torch.from_numpy(rewards) * alive).sum(-1)
        for k, v in (('obs', obs), ('alive', alive), ('hears', hears), ('action', action), ('log_prob', log_prob),
                     ('acting', torch.from_numpy(stepped)), ('reward', reward * torch.from_numpy(stepped)),
                     ('continues', torch.from_numpy(stepped & (~ended | cut)))):
            rows[k].append(v)
        reached |= terms & stepped[:, None]
        length += stepped
        bootstrap, running = cut, running & ~ended
    batch = {k: torch.stack(v) for k, v in rows.items()}
    stats = {'success': reached.all(1), 'return': batch['reward'].sum(0).numpy(), 'length': length}
    return batch, stats

### Training

`actor_critic_losses` replays the episodes through the agents from their start and computes the
critic's TD loss, `(r_t + gamma Q̂_{t+1} - Q̂_t)^2` (the next row's `Q̂` when the episode continues
or bootstraps, held fixed), and the policy loss `-log π(a_i^t | h_i^t) Q̂_t` (held fixed; minus the
batch's mean `Q̂` with `q_baseline`) of the agents acting, minus the entropy bonus.

In [ ]:
#| export
def actor_critic_losses(
    model: TarMAC,
    batch: dict[str, torch.Tensor], # Rows of `rollout` (L, E, ...)
    gamma: float = 0.99,
    entropy_coef: float = 0.01,
    critic_coef: float = 1.0,
    q_baseline: bool = False, # Subtract the batch's mean Q̂ from the policy gradient's weights (not in the paper)
) -> dict[str, torch.Tensor]:
    "`loss` (to minimize), `policy_loss`, `critic_loss`, `entropy` and `q` of a batch of episodes."
    L, E, N = batch['action'].shape
    h, c = model.initial_state(E, batch['obs'].device)
    logits, q = [], []
    for t in range(L):
        lt, h, c = model.step(batch['obs'][t], batch['alive'][t], batch['hears'][t], h, c)
        logits.append(lt)
        q.append(model.q_value(h, batch['action'][t], batch['alive'][t]))
    logits, q = torch.stack(logits), torch.stack(q)                     # (L, E, N, A), (L, E)
    acting = batch['acting'].float()                                    # (L, E): rows whose actions were taken
    next_q = torch.cat([q[1:], torch.zeros_like(q[:1])]).detach()
    target = batch['reward'] + gamma * batch['continues'].float() * next_q
    critic_loss = ((target - q).pow(2) * acting).sum() / acting.sum().clamp(min=1)
    weight = q.detach() - (q.detach()[acting.bool()].mean() if q_baseline else 0.0)
    agents = (batch['alive'] & batch['acting'].unsqueeze(-1)).float()   # (L, E, N): agents that acted
    dist = torch.distributions.Categorical(logits=logits)
    n = agents.sum().clamp(min=1)
    policy_loss = -(dist.log_prob(batch['action']) * weight.unsqueeze(-1) * agents).sum() / n
    entropy = (dist.entropy() * agents).sum() / n
    return {'loss': policy_loss + critic_coef * critic_loss - entropy_coef * entropy, 'policy_loss': policy_loss.detach(),
            'critic_loss': critic_loss.detach(), 'entropy': entropy.detach(), 'q': q.detach()[acting.bool()].mean()}


def train_tarmac(
    model: TarMAC,
    world, # A `World`: every update plays one episode per env (TarMAC: batches of 16)
    episodes: int = 20_000, # Episodes played in total (updates: episodes / world.num_envs)
    lr: float = 7e-4, # RMSProp (TarMAC: 7e-4, alpha 0.99)
    gamma: float = 0.99,
    entropy_coef: float = 0.01,
    critic_coef: float = 1.0,
    q_baseline: bool = False,
    max_grad_norm: float | None = None, # Clip the gradient norm (not in the paper)
    seed: int = 0, # Seeds of the episodes: seed, seed + 1, ...
    log_every: int = 20, # Updates between two `log` calls
    log: Callable[[dict], None] | None = None, # Called with the episodes played and the means since the last call
) -> TarMAC:
    "Train `model` online with TarMAC's multi-agent actor-critic; returns it in eval mode."
    device = next(model.parameters()).device
    optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, alpha=0.99)
    E = world.num_envs
    totals: dict[str, float] = {}
    count = 0
    for update in range(1, episodes // E + 1):
        model.eval()
        batch, stats = rollout(model, world, seed + (update - 1) * E)
        model.train()
        losses = actor_critic_losses(model, {k: v.to(device) for k, v in batch.items()}, gamma, entropy_coef,
                                     critic_coef, q_baseline)
        optimizer.zero_grad(set_to_none=True)
        losses['loss'].backward()
        if max_grad_norm is not None:
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        row = {**{k: float(v) for k, v in losses.items()}, 'success_rate': float(stats['success'].mean()),
               'episode_return': float(stats['return'].mean()), 'length': float(stats['length'].mean())}
        for k, v in row.items():
            totals[k] = totals.get(k, 0.0) + v
        count += 1
        if log is not None and (update % log_every == 0 or update == episodes // E):
            log({'episodes': update * E, **{k: v / count for k, v in totals.items()}})
            totals, count = {}, 0
    model.reset()
    return model.eval()

### Saving and loading

In [ ]:
#| export
@patch
def save(self: TarMAC, path: str | Path, **extra: Any) -> Path:
    "Save the config, the weights and `extra` (e.g. `env_name`, `env_kwargs`) to `path` (e.g. `tarmac.pt`)."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'config': self.config, 'state_dict': self.state_dict(), **extra}, path)
    return path


@patch(cls_method=True)
def load(cls: TarMAC, path: str | Path, map_location: Any = 'cpu') -> TarMAC:
    "A `TarMAC` saved with `save`; what it stored besides the model is in `checkpoint`."
    checkpoint = torch.load(path, map_location=map_location, weights_only=True)
    model = cls(**checkpoint.pop('config'))
    model.load_state_dict(checkpoint.pop('state_dict'))
    model.checkpoint = checkpoint
    return model.eval()

### Tests

Rollouts on a small FindGoal (episodes end once, when every agent is done; a bootstrap row after
the time limit; replaying them gives the actions' probabilities of acting), a few updates, acting in
a `World` with 1 and 2 rounds, messages over comm-core, and loading back.

In [ ]:
#| hide
import tempfile
import comm_core as cc
from stable_marl import World
from stable_marl.policy import FeedForwardPolicy

kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=12)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=4, goal_conditioned=True, **kw)
world.reset(seed=list(range(4)))
dim = commnet_features({k: v[:, -1] for k, v in world.infos.items() if not k.startswith('_')}, grid_size=7).shape[-1]
torch.manual_seed(0)
model = TarMAC(dim, 2, hidden_dim=32, grid_size=7)
batch, stats = rollout(model, world, seed=0)
L = len(batch['action'])
test_eq(batch['acting'].sum(0).numpy(), stats['length'])                # rows played, per env
for e in range(4):
    played = int(stats['length'][e])
    if stats['success'][e]:
        assert not batch['continues'][played - 1, e]                     # the goal: no continuation
    else:
        assert batch['continues'][played - 1, e] and batch['alive'][played, e].any()   # the time limit: a bootstrap row
        assert not batch['acting'][played, e]
    assert not batch['acting'][played:, e].any()

# replaying the episodes with the recorded masks (as training does) gives the probabilities they were played with
h, c = model.initial_state(4)
with torch.no_grad():
    for t in range(L):
        logits, h, c = model.step(batch['obs'][t], batch['alive'][t], batch['hears'][t], h, c)
        played = batch['alive'][t] & batch['acting'][t].unsqueeze(-1)
        replayed = torch.distributions.Categorical(logits=logits).log_prob(batch['action'][t])
        test_close(replayed[played], batch['log_prob'][t][played], eps=1e-5)
losses = actor_critic_losses(model, batch)
assert all(torch.isfinite(v).all() for v in losses.values())
logs = []
train_tarmac(model, world, episodes=12, log_every=1, log=logs.append)
test_eq([r['episodes'] for r in logs], [4, 8, 12])

world.set_policy(FeedForwardPolicy(model))
test_eq(len(world.evaluate(episodes=4, seed=100)['episode_successes']), 4)
two = TarMAC(dim, 2, hidden_dim=32, grid_size=7, rounds=2, latent_bits=4)
train_tarmac(two, world, episodes=4)
world.set_policy(FeedForwardPolicy(two))
world.evaluate(episodes=2, seed=100)

# messages over comm-core: each round's [k, v] bundle, of `message_bits`
two.links = NetworkLinks(2, channel=cc.ErasureChannel(loss_rate=0.0))
world.evaluate(episodes=2, seed=100)
m = two.links.metrics[0]
assert m.transmissions > 0; test_eq(m.bits_sent, m.transmissions * two.message_bits)
two.links = NetworkLinks(2, channel=cc.ErasureChannel(loss_rate=1.0))
batch, _ = rollout(two, world, seed=0)
test_eq((batch['hears'] & ~torch.eye(2, dtype=torch.bool)).any(), False)   # nothing but each agent's own
two.links = None

path = model.save(f'{tempfile.mkdtemp()}/tarmac.pt', env_name='MultiGrid-FindGoal-15x15-v0', env_kwargs=kw)
again = TarMAC.load(path)
test_eq(again.checkpoint['env_kwargs'], kw)
h, c = model.initial_state(4)
test_close(again.step(batch['obs'][0], batch['alive'][0], batch['hears'][0], h, c)[0],
           model.step(batch['obs'][0], batch['alive'][0], batch['hears'][0], h, c)[0])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()